In [1]:
# toy_ngafid.py
# Create a toy NGAFID-like dataset for quick MAE prototyping (no large downloads).

from __future__ import annotations
import math, random, os
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, Features, Value

# ----------------------------
# 1) Schema (edit as you like)
# ----------------------------
# Typical NGAFID-style columns
COLUMNS = [
    "timestamp",  # string timestamp or seconds since start
    "altmsl",     # altitude (ft)
    "ias",        # indicated airspeed (knots)
    "vspd",       # vertical speed (ft/min)
    "pitch",      # degrees
    "roll",       # degrees
    "hdg",        # heading (deg)
    # Add any extras you might see in shards:
    # "VAL", "VCDI",
]

# Hugging Face feature types (keep timestamp as string; rest float32)
HF_FEATURES = Features({
    "timestamp": Value("string"),
    **{c: Value("float32") for c in COLUMNS if c != "timestamp"},
})

# --------------------------------------
# 2) Random generators (realistic-ish)
# --------------------------------------
def sample_flight(n_rows: int, start_time_s: int = 0) -> pd.DataFrame:
    """
    Create a single synthetic flight time series with n_rows.
    Values are crude but bounded to plausible avionics ranges.
    """
    t = np.arange(start_time_s, start_time_s + n_rows, dtype=np.int64)

    # altitude: climb, cruise, descent pattern
    climb = np.clip(np.linspace(0, 12000, n_rows//3), 0, None)
    cruise = np.full(n_rows//3, 12000.0)
    descent = np.clip(np.linspace(12000, 1000, n_rows - 2*(n_rows//3)), 0, None)
    alt = np.concatenate([climb, cruise, descent])
    alt += np.random.normal(0, 80, size=n_rows)

    # airspeed: ramp up then stable with noise (knots)
    ias = np.clip(140 + 20*np.tanh(np.linspace(-2, 2, n_rows)) + np.random.normal(0, 4, n_rows), 60, 220)

    # vertical speed (ft/min): noisy, around 0 in cruise
    vspd = np.random.normal(0, 200, n_rows)
    vspd[: n_rows//3] += 800   # climb bias
    vspd[-n_rows//3 :] -= 800  # descent bias
    vspd = np.clip(vspd, -2000, 2000)

    # pitch/roll in degrees
    pitch = np.clip(np.random.normal(2, 3, n_rows), -10, 15)   # gentle
    roll  = np.clip(np.random.normal(0, 10, n_rows), -45, 45)  # bank
    # heading: slow random walk in [0, 360)
    hdg = np.cumsum(np.random.normal(0, 1, n_rows)) % 360

    df = pd.DataFrame({
        "timestamp": pd.to_datetime(t, unit="s").astype(str),
        "altmsl": alt.astype("float32"),
        "ias": ias.astype("float32"),
        "vspd": vspd.astype("float32"),
        "pitch": pitch.astype("float32"),
        "roll": roll.astype("float32"),
        "hdg": hdg.astype("float32"),
    })

    return df

# --------------------------------------------------------
# 3) Build a toy dataset (in-memory and/or write shards)
# --------------------------------------------------------
def make_toy_ngafid_dataset(
    num_flights: int = 20,
    rows_per_flight: Tuple[int, int] = (2000, 6000),  # flights vary in length
    write_shards_like_hf: bool = False,
    out_dir: str | Path = "toy_ngafid",
) -> DatasetDict:
    """
    Create a toy dataset with multiple flights. Returns a DatasetDict with a 'train' split.
    If write_shards_like_hf=True, also writes CSV shards to toy_ngafid/flights/*.csv to mirror HF layout.
    """
    rng = np.random.default_rng(0)

    all_frames: List[pd.DataFrame] = []
    out_dir = Path(out_dir)
    flights_dir = out_dir / "flights"
    if write_shards_like_hf:
        flights_dir.mkdir(parents=True, exist_ok=True)

    for i in range(num_flights):
        n_rows = int(rng.integers(rows_per_flight[0], rows_per_flight[1]))
        df = sample_flight(n_rows, start_time_s=i * 10_000)
        # Optional light missingness (simulate sensor dropouts)
        for col in ["altmsl", "ias", "vspd", "pitch", "roll", "hdg"]:
            mask = rng.random(n_rows) < 0.005  # 0.5% NaNs
            df.loc[mask, col] = np.nan

        if write_shards_like_hf:
            # write a shard named like the real dataset
            shard_path = flights_dir / f"flight_{i:05d}.csv"
            df.to_csv(shard_path, index=False)

        all_frames.append(df)

    big_df = pd.concat(all_frames, ignore_index=True)

    # Ensure only the declared columns exist and types match
    big_df = big_df[[c for c in COLUMNS]]  # drop any extras accidentally added

    ds = Dataset.from_pandas(big_df, features=HF_FEATURES, preserve_index=False)
    return DatasetDict({"train": ds})

# -----------------------
# 4) Quick usage example
# -----------------------
if __name__ == "__main__":
    toy = make_toy_ngafid_dataset(
        num_flights=10,
        rows_per_flight=(1500, 3000),
        write_shards_like_hf=True,   # set False if you only want in-memory HF dataset
        out_dir="toy_ngafid",
    )

    print(toy)               # DatasetDict with 'train'
    print(toy.shape)
    print(toy["train"])      # size (rows) and columns
    print(toy["train"].shape)      # size (rows) and columns
    print(toy["train"][0])   # one row sample
    # You can now plug toy["train"] into your MAE dataloader.


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['timestamp', 'altmsl', 'ias', 'vspd', 'pitch', 'roll', 'hdg'],
        num_rows: 23880
    })
})
{'train': (23880, 7)}
Dataset({
    features: ['timestamp', 'altmsl', 'ias', 'vspd', 'pitch', 'roll', 'hdg'],
    num_rows: 23880
})
(23880, 7)
{'timestamp': '1970-01-01 00:00:00', 'altmsl': 70.70861053466797, 'ias': 120.56128692626953, 'vspd': 806.134033203125, 'pitch': 1.0467193126678467, 'roll': -4.694386959075928, 'hdg': 0.34959855675697327}


In [ ]:
# Discover (and optionally reuse) the original feature column names
from datasets import load_dataset
from itertools import islice

ds = load_dataset(
    "csv",
    data_files="hf://datasets/username/NGAFID-LOCI-GATS-Data/preprocessed_data/train/*.csv",
    streaming=True,
)["train"]

all_cols = set()
N = 2000  # scan up to N rows
for row in islice(ds, N):
    all_cols.update(row.keys())

# Columns that are NOT model features (adjust if your CSV has different auxiliaries)
NON_FEATURE_COLS = {"timestamp", "flight_id", "split", "label", "path", "index", "idx"}

# Keep at most 44 feature columns (the real set should already be 44)
feature_cols = [c for c in sorted(all_cols) if c not in NON_FEATURE_COLS]
feature_cols = feature_cols[:44]

print("Total discovered columns:", len(all_cols))
print("Selected feature columns (up to 44):", len(feature_cols))
print(feature_cols)


In [1]:
import numpy as np
from datasets import Dataset, DatasetDict, Features, Array3D, Sequence, Value
from datasets import load_from_disk

# Target shape per row: [C, T, F] = [1, 10000, 44]; 8 rows total
BATCH, C, T, F = 8, 1, 10000, 44

# If Cell 1 discovered real names, reuse them; else placeholders
F_NAMES = feature_cols if "feature_cols" in globals() and len(feature_cols) == F else [f"feat_{i:02d}" for i in range(F)]

rng = np.random.default_rng(0)

def make_example(T: int, F: int) -> np.ndarray:
    steps = rng.normal(0.0, 1.0, size=(T, F)).cumsum(axis=0) / 50.0
    return steps.astype("float32")[None, ...]  # [1, T, F]

inputs = [make_example(T, F) for _ in range(BATCH)]

FEATURES = Features({
    "input": Array3D(shape=(C, T, F), dtype="float32"),
    "feature_names": Sequence(Value("string")),
})

toy_train = Dataset.from_dict(
    {"input": inputs, "feature_names": [F_NAMES] * BATCH},
    features=FEATURES,
)
toy = DatasetDict({"train": toy_train})
print(toy)

# ---- Save locally ----
save_dir = "toy_ngafid_loci_gats"   # change this path if you want
toy.save_to_disk(save_dir)
print(f"✅ Saved toy dataset to: {save_dir}")

# ---- (Optional) quick reload + shape check ----
toy2 = load_from_disk(save_dir).with_format("numpy", columns=["input"], output_all_columns=True)
ex0 = toy2["train"][0]["input"]
print("Single example shape (should be 1x10000x44):", ex0.shape)
batch_stack = np.stack([toy2["train"][i]["input"] for i in range(len(toy2["train"]))], axis=0)
print("Stacked batch shape (should be 8x1x10000x44):", batch_stack.shape)


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['input', 'feature_names'],
        num_rows: 8
    })
})


Saving the dataset (1/1 shards): 100%|██████████| 8/8 [00:00<00:00, 467.79 examples/s]

✅ Saved toy dataset to: toy_ngafid_loci_gats
Single example shape (should be 1x10000x44): (1, 10000, 44)
Stacked batch shape (should be 8x1x10000x44): (8, 1, 10000, 44)
